# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** _(fill in)_
**Student ID:** _(fill in)_
**Group:** _(fill in)_
**Repository:** https://github.com/abylajamankeldy/css4007-hw3

## AI tool disclosure

- **Claude (Anthropic)** helped me write the code cells, set up the notebook and work out Part A step by step. Part A arithmetic was also re-checked by code (the check cell below).
- **Model in Part C:** I had no OpenAI key, so every Part C call uses **`openai/gpt-oss-120b` through Groq's OpenAI-compatible API** (the cell that creates the client prints the provider and model name). The README names `gpt-5.6-luna`; the three systems still use the same model, so the comparison between them is fair, but the absolute answers may differ from what `gpt-5.6-luna` would give.
- The written analysis in Parts B and C and the final questions quotes the outputs printed by this notebook on my own run.
- No API key is in this notebook or in the repository (`.env` is gitignored).

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · generate · text · and · answer · questions | 9 |
| D2 | retrieval · augmented · generation · retrieves · relevant · documents · before · generating · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** `how`, `does`, `rag`, `retrieve`, `information`  (the query lowercased, no stop words removed, no stemming)

**TF, DF and IDF of each query term** (N = 3). The idf formula is `ln((N − n + 0.5) / (n + 0.5))`:

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `does`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retrieve`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `information`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Every query term appears in **zero** documents. The word `retrieve` is not `retrieves` and not `retrieval` — without stemming these are three different tokens. `rag` is never written out in D2 (only the words "Retrieval-Augmented Generation" are), and `does`, `how`, `information` occur nowhere.

**BM25 of each document** (a term contributes only if f(t, d) > 0, because the numerator `idf · f · (k1+1)` is then 0):

> - **D1:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0, `information` f=0 → score = 0 + 0 + 0 + 0 + 0 = **0**
> - **D2:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0 (D2 has `retrieves` and `retrieval`, not `retrieve`), `information` f=0 → score = **0**
> - **D3:** all five f=0 → score = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

The three-way tie at zero is not an arithmetic mistake. I rechecked: no query token is equal to any document token, so BM25 has nothing to rank by. BM25 cannot pick a first document here.

## A1 (b) — BM25 with the stems from README §3

Stems used: retrieve / retrieves / retrieval → `retriev`; generate / generating / generation → `gener`; information → `inform`; does → `doe`. Every other word stays as it is.

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · gener · text · and · answer · questions | 9 |
| D2 | retriev · augmented · gener · retriev · relevant · documents · before · gener · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**  (stemming does not change the number of tokens)

**Stemmed query terms:** `how`, `doe`, `rag`, `retriev`, `inform`

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `doe`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retriev`: n(t) = 1 → idf = ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) = ln(1.6667) = **0.5108**
- `inform`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Only `retriev` now matches, and only in D2 (twice: `retrieval` and `retrieves`). D1 and D3 contain no query term (D1 has `gener`, which is not a query term).

**BM25 of each document**, worked:

> **D1** = 0 (none of `how`, `doe`, `rag`, `retriev`, `inform` occurs in D1)
>
> **D2** = only the term `retriev` contributes:
>
> - **D2**, term `retriev`: idf = 0.5108, f = 2, |d| = 10, avgdl = 8.3333
>   - length part: 1 − b + b·|d|/avgdl = 1 − 0.75 + 0.75·10/8.3333 = 1.1500
>   - denominator: f + k1·(that) = 2 + 1.5·1.1500 = 3.7250
>   - numerator: idf·f·(k1+1) = 0.5108·2·2.5 = 2.5541
>   - term score = 2.5541 / 3.7250 = **0.6857**
>
> **D3** = 0 (no query term occurs in D3)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> - **(a):** none. All three documents score 0, so BM25 has no first document. The query asks about "retrieve", but D2 says "retrieves" and "retrieval", and without stemming a token has to be identical to count. The result looked like a mistake, I checked every term again, and it survives the check, so it is the real behaviour of exact-token matching.
> - **(b):** **D2**, score 0.6857. After stemming, `retrieves` and `retrieval` both become `retriev`, so the query term `retriev` is found twice in D2. D1 and D3 still score 0.
> - The only thing that changed between (a) and (b) is the tokenisation, not the documents or the formula. D2 is also the document a human would pick, but BM25 got there only through shared word stems, never through meaning. Note that `rag`, `how`, `doe` and `inform` still match nothing: D2 never writes "RAG" or "information".

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

Formula: cosine(Q, D) = (Q · D) / (‖Q‖ · ‖D‖), with Q · D = Σ qᵢ·dᵢ and ‖v‖ = √(Σ vᵢ²).

- ‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644
- ‖D1‖ = √(0.7² + 0.1² + 0.4²) = √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124
- ‖D2‖ = √(0.9² + 0.3² + 0.6²) = √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225
- ‖D3‖ = √(0.1² + 0.8² + 0.2²) = √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307
- cos(Query, D1) = 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956**
- cos(Query, D2) = 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977**
- cos(Query, D3) = 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (cosine 0.9977). D1 is very close behind (0.9956), and D3 is far away (0.4244): D3's vector points mostly along the second axis, the query's mostly along the first.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> The cosine ranking is **D2 > D1 > D3**.
> - Against **A1 (a)**: not comparable — BM25 gave a three-way tie at 0, so it said nothing; cosine produces a full ranking.
> - Against **A1 (b)**: the **top document is the same (D2)**. The rest differs: BM25 (b) leaves D1 and D3 tied at 0, whereas cosine puts D1 clearly second (0.9956) and D3 last (0.4244). So the two methods agree on the winner after stemming, and cosine also tells how close D1 is.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 counts identical tokens. The query words `retrieve` and `information` are not in any document as written, so in (a) BM25 sees no overlap at all; even in (b) it only finds `retriev` in D2 and never sees that D1 ("answer questions") is also about the same topic. An embedding model compares meaning: "How does RAG retrieve information?" is close to "Retrieval-Augmented Generation retrieves relevant documents…" because RAG *is* retrieval-augmented generation, a fact BM25 cannot know from the letters `rag`. It is also close to D1 (language models that answer questions), because both are about question answering, which shares no words with the query. The weakness is the opposite one: embeddings blur exact strings (a form code such as GR-305), where BM25 is exact.
> (Here the vectors were given, so the "meaning" is already baked into the numbers. D1 and D2 are both near the query because their vectors point in nearly the same direction.)

Check of the Part A arithmetic (the working above is written out by hand; this cell only confirms the numbers):

In [ ]:
import math
k1, b = 1.5, 0.75
D = {"D1": "large language models can generate text and answer questions".split(),
     "D2": "retrieval augmented generation retrieves relevant documents before generating an answer".split(),
     "D3": "embeddings represent text as numerical vectors".split()}
STEM = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev", "generate": "gener",
        "generating": "gener", "generation": "gener", "information": "inform", "does": "doe"}
query = "how does rag retrieve information".split()

def bm25_by_hand(docs, q):
    N = len(docs); avg = sum(len(w) for w in docs.values()) / N
    scores = {}
    for d, w in docs.items():
        s = 0
        for t in q:
            n = sum(1 for ww in docs.values() if t in ww)
            idf = math.log((N - n + 0.5) / (n + 0.5))
            f = w.count(t)
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(w) / avg))
        scores[d] = round(s, 4)
    return scores

print("(a) as written :", bm25_by_hand(D, query))
Ds = {d: [STEM.get(x, x) for x in w] for d, w in D.items()}
print("(b) with stems :", bm25_by_hand(Ds, [STEM.get(x, x) for x in query]))

Q = [0.8, 0.2, 0.5]
V = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
norm = lambda v: math.sqrt(sum(x * x for x in v))
for d, v in V.items():
    dot = sum(x * y for x, y in zip(Q, v))
    print(d, "dot =", round(dot, 2), "|Q| =", round(norm(Q), 4), "|D| =", round(norm(v), 4), "cos =", round(dot / (norm(Q) * norm(v)), 4))

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [ ]:
import json, re, os, math
import numpy as np
from dotenv import load_dotenv
load_dotenv()          # reads OPENAI_API_KEY / GROQ_API_KEY / HF_TOKEN from .env (never printed)

corpus  = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa      = json.load(open("data/qa.json", encoding="utf-8"))["questions"]
ids     = [c["id"] for c in corpus]
by_id   = {c["id"]: c for c in corpus}
print(len(corpus), "chunks,", len(queries), "fixed queries,", len(qa), "questions")

## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase, then `re.findall(r"\w+", text)` (a token is a run of letters or digits; the regex is Unicode-aware, so Kazakh and Russian words are kept whole). **No stemming and no stop-word removal**, so `retrieve` and `retrieves` stay different tokens, as in A1 (a). A form code such as `GR-305` becomes two tokens, `gr` and `305`, because the hyphen splits it. `BM25Okapi(k1=1.5, b=0.75)`.

In [ ]:
from rank_bm25 import BM25Okapi

def tok(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tok(c["text"]) for c in corpus], k1=1.5, b=0.75)

def search_bm25(q, k=3):
    s = bm25.get_scores(tok(q))
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

def first_line(cid, n=80):
    c = by_id[cid]
    return (c["title"] + " | " + c["text"].split("\n")[0])[:n]

print("example:", search_bm25(queries[0]["text"]))

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")          # default float32; token is read from HF_TOKEN
doc_emb = model.encode_document([c["text"] for c in corpus], show_progress_bar=False)
print("document embeddings:", doc_emb.shape, doc_emb.dtype)        # these vectors ARE my "vector store": one numpy array

def search_emb(q, k=3):
    qe = model.encode_query(q)
    s = np.asarray(model.similarity(qe, doc_emb)[0])
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

print("example:", search_emb(queries[0]["text"]))

## My own queries

Written **before** running any retriever. `relevant` is decided from reading the handbook, not from results.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which office is in room 214? | GO-21 | **BM25.** The query is built around the exact token `214` (and `room`), which appears in only one chunk. An embedding model turns numbers into a blur, and "office" could pull any chunk about the Grant Office. |
| Q-10 | Какие документы подтверждают доход семьи? | GO-10 | **Embeddings.** The question is in Russian ("which documents confirm the family's income") while the answer chunk GO-10 ("Proof of income … salary certificate …") is in English. BM25 shares no words across the two languages; a multilingual embedding model should still match the meaning. |

In [ ]:
own = [
    {"id": "Q-09", "text": "Which office is in room 214?",                "relevant": ["GO-21"]},
    {"id": "Q-10", "text": "Какие документы подтверждают доход семьи?",    "relevant": ["GO-10"]},
]
allq = queries + own
print(len(allq), "queries in total")

## Results for all ten or more queries: top 3 from each retriever

In [ ]:
def hit(res, relevant):
    return any(i in relevant for i, _ in res)

results = {}
for q in allq:
    b, e = search_bm25(q["text"]), search_emb(q["text"])
    results[q["id"]] = {"bm25": b, "emb": e, "hb": hit(b, q["relevant"]), "he": hit(e, q["relevant"])}
    print(f"{q['id']}  {q['text']}")
    print(f"   relevant: {q['relevant']}")
    for name, res, h in (("BM25", b, results[q['id']]['hb']), ("EmbeddingGemma", e, results[q['id']]['he'])):
        print(f"   {name}   hit@3 = {'YES' if h else 'no'}")
        for r, (cid, sc) in enumerate(res, 1):
            mark = "*" if cid in q["relevant"] else " "
            print(f"     {r}. {cid}{mark} {sc:8.4f}  {first_line(cid)}")
    print()
print("(* = chunk is in `relevant`)")

## B3 — comparison

The table below is printed by the next cell from the same `results` dictionary as the top-3 lists above. The three examples are written after it.

In [ ]:
from IPython.display import Markdown, display
rows = ["| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |", "|---|---|:---:|---|:---:|"]
for q in allq:
    r = results[q["id"]]
    rows.append(f"| {q['id']} | {', '.join(i for i, _ in r['bm25'])} | {'✓' if r['hb'] else '✗'} | {', '.join(i for i, _ in r['emb'])} | {'✓' if r['he'] else '✗'} |")
tb = sum(r["hb"] for r in results.values()); te = sum(r["he"] for r in results.values())
rows.append(f"| **hit@3 total** | | **{tb}/{len(allq)}** | | **{te}/{len(allq)}** |")
display(Markdown("\n".join(rows)))

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [ ]:
import time, tiktoken
from openai import OpenAI

enc = tiktoken.get_encoding("o200k_base")
def ntok(s): return len(enc.encode(s))

if os.getenv("OPENAI_API_KEY"):
    client, MODEL, GROQ = OpenAI(), "gpt-5.6-luna", False
elif os.getenv("GROQ_API_KEY"):
    client, MODEL, GROQ = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1"), "openai/gpt-oss-120b", True
else:
    raise SystemExit("put OPENAI_API_KEY or GROQ_API_KEY in .env")
print("provider:", "Groq (OpenAI-compatible API)" if GROQ else "OpenAI", "| model:", MODEL)

_last = [0.0]
def ask(messages):
    """one chat call -> (answer text, prompt_tokens from usage, completion_tokens)"""
    kw = {"max_tokens": 4000, "temperature": 0} if GROQ else {}
    for attempt in range(6):
        wait = 2.5 - (time.time() - _last[0])          # stay under the free-tier rate limit
        if GROQ and wait > 0: time.sleep(wait)
        _last[0] = time.time()
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages, **kw)
            return (r.choices[0].message.content or "").strip(), r.usage.prompt_tokens, r.usage.completion_tokens
        except Exception as ex:
            if attempt == 5: raise
            print("   retry after error:", type(ex).__name__); time.sleep(5 * (attempt + 1))

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def context_block(chunk_ids):
    return "\n\n".join(f"[{n}] ({cid} | {by_id[cid]['title']})\n{by_id[cid]['text']}" for n, cid in enumerate(chunk_ids, 1))

def rag_messages(question, chunk_ids):
    user = "Context:\n" + context_block(chunk_ids) + "\n\nQuestion: " + question
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

## C1 — the complete prompt, printed before it is sent

In [ ]:
q1 = next(x for x in qa if x["id"] == "C-02")
top = search_emb(q1["question"], 3)
c1_ids = [cid for cid, _ in top]
print("EmbeddingGemma top 3:", top, "\n")

msgs = rag_messages(q1["question"], c1_ids)
print("=" * 30, "WHAT THE MODEL RECEIVES", "=" * 30)
for m in msgs:
    print(f"--- role: {m['role']} ---")
    print(m["content"])
    print()
print("=" * 85)

# token accounting with tiktoken o200k_base
sys_t  = ntok(SYSTEM)
q_t    = ntok("Question: " + q1["question"])
ctx_text = "Context:\n" + context_block(c1_ids)
ctx_t  = ntok(ctx_text)
total_parts = sys_t + q_t + ctx_t
whole = ntok(msgs[0]["content"]) + ntok(msgs[1]["content"])

answer, prompt_tokens, completion_tokens = ask(msgs)
print("answer:", answer, "\n")

from IPython.display import Markdown, display
display(Markdown(f"""| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | {sys_t} |
| Question (with the label "Question:") | {q_t} |
| Retrieved context (with the label "Context:" and the block headers) | {ctx_t} |
| **Total input** (sum of the three) | **{total_parts}** |
| **`usage.prompt_tokens` from the response** | **{prompt_tokens}** |"""))
print("tiktoken on the two message strings as a whole:", whole)
print("difference usage - tiktoken sum:", prompt_tokens - total_parts)
per_chunk = [ntok(by_id[c]["text"]) for c in c1_ids]
print("number of retrieved chunks:", len(c1_ids))
print("tokens per chunk (text only):", dict(zip(c1_ids, per_chunk)))
print("total context size:", ctx_t, "tokens,", len(ctx_text), "characters  (chunk text only:", sum(per_chunk), "tokens,", sum(len(by_id[c]["text"]) for c in c1_ids), "characters)")

The table above is printed by the cell above it (from `ntok(...)` and `response.usage`), not typed. The numbers for the discussion are quoted in the answers below.

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>

## C2 — three systems, six questions

In [ ]:
runs = {}
for q in qa:
    qid = q["id"]
    b_ids = [cid for cid, _ in search_bm25(q["question"], 3)]
    e_ids = [cid for cid, _ in search_emb(q["question"], 3)]
    a_llm, pt_llm, _ = ask([{"role": "user", "content": q["question"]}])            # system 1: no context
    a_bm,  pt_bm,  _ = ask(rag_messages(q["question"], b_ids))                        # system 2: BM25 top 3
    a_em,  pt_em,  _ = ask(rag_messages(q["question"], e_ids))                        # system 3: embedding top 3
    runs[qid] = {"llm": a_llm, "bm25": a_bm, "emb": a_em, "bm25_ids": b_ids, "emb_ids": e_ids,
                 "pt": {"llm": pt_llm, "bm25": pt_bm, "emb": pt_em}}
    print("=" * 100)
    print(qid, "|", q["question"])
    print("REFERENCE:", q["reference"])
    print("relevant chunks:", q["relevant"])
    print(f"\n[LLM only]   (prompt_tokens {pt_llm})\n{a_llm}")
    print(f"\n[BM25 RAG]   chunks {b_ids}  (prompt_tokens {pt_bm})\n{a_bm}")
    print(f"\n[Embedding RAG] chunks {e_ids}  (prompt_tokens {pt_em})\n{a_em}")

The answers above are the evidence; the table below marks each one against `reference`.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [ ]:
def show_case(qid):
    q = next(x for x in qa if x["id"] == qid); r = runs[qid]
    print(qid, "|", q["question"]); print("REFERENCE:", q["reference"]); print("relevant:", q["relevant"], "\n")
    print("1. BM25 top 3:       ", [(c, round(s, 3)) for c, s in search_bm25(q["question"], 3)])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in search_emb(q["question"], 3)])
    for name, key in (("BM25", "bm25_ids"), ("EmbeddingGemma", "emb_ids")):
        print(f"\n3. Context sent to the model by {name} RAG:")
        print(context_block(r[key]))
    print("\nAnswers:")
    for name, key in (("LLM only", "llm"), ("BM25 RAG", "bm25"), ("Embedding RAG", "emb")):
        print(f"- {name}: {r[key]}")

show_case("C-03")

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [ ]:
show_case("C-06")

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [ ]:
show_case("C-01")

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>

In [ ]:
# K = 1, 3, 8 with EmbeddingGemma on two questions
for qid in ("C-03", "C-01"):
    q = next(x for x in qa if x["id"] == qid)
    print("=" * 100); print(qid, "|", q["question"]); print("REFERENCE:", q["reference"])
    for K in (1, 3, 8):
        cids = [c for c, _ in search_emb(q["question"], K)]
        a, pt, _ = ask(rag_messages(q["question"], cids))
        print(f"\n--- K = {K}: chunks {cids}, relevant found: {[c for c in cids if c in q['relevant']]}, prompt_tokens {pt}")
        print(a)

**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>